In [5]:
import zipfile
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical


In [12]:
import zipfile
import cv2
import numpy as np
from io import BytesIO

# Function to load images and labels from a zip file
def load_images_from_zip(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        # Get the list of all files in the zip file
        image_files = [f for f in zip_ref.namelist() if not f.endswith('/') and '/.' not in f]  # Filter valid files
        X, y = [], []
        
        # Loop through the files and extract images and labels
        for file_name in image_files:
            with zip_ref.open(file_name) as file:
                img_data = file.read()
                img = cv2.imdecode(np.frombuffer(img_data, np.uint8), cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (64, 64))  # Resize images
                X.append(img)
                
                # Extract label from the folder structure (e.g., 'train/0/image_name.png' or 'train/1/image_name.png')
                split_path = file_name.split('/')
                if len(split_path) >= 3:  # Ensure valid paths
                    label = split_path[1]  # Folder name (0 or 1) is the label
                    y.append(label)
        
        # Convert X to a numpy array and normalize pixel values
        X = np.array(X) / 255.0  # Normalize pixel values to [0, 1]
        return X, np.array(y)

# Load images and labels directly from the zip file
zip_path = r'D:\Medical imaging diagnostics(breast cancer)\archive (9).zip'   # Update this with your zip path
X, y = load_images_from_zip(zip_path)

# Reshape images to flatten them for training a classifier
X = X.reshape(X.shape[0], -1)  # Flatten images into 1D vectors

# Display some stats
print(f"Number of images: {len(X)}")
print(f"Shape of each image: {X[0].shape}")
print(f"Number of labels: {len(y)}")
print(f"Unique labels: {set(y)}")


Number of images: 3383
Shape of each image: (4096,)
Number of labels: 3383
Unique labels: {'1', '0'}


In [14]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [15]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)


In [16]:
#Logistic Regression Model (Manually Implemented)

# Initialize weights and bias
weights = np.zeros(X_train.shape[1])
bias = 0
learning_rate = 0.01
epochs = 1000


In [17]:
# Sigmoid activation function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


In [18]:
# Predict function
def predict(X):
    z = np.dot(X, weights) + bias
    return sigmoid(z)


In [23]:
# Mini-batch Gradient Descent Implementation with L2 Regularization
def train_logistic_regression_with_mini_batch(X_train, y_train, learning_rate, epochs, batch_size, lambda_reg):
    global weights, bias
    m = len(y_train)
    
    for epoch in range(epochs):
        # Shuffle data for each epoch
        indices = np.random.permutation(m)
        X_train_shuffled = X_train[indices]
        y_train_shuffled = y_train[indices]
        
        # Mini-batch gradient descent
        for i in range(0, m, batch_size):
            # Get the mini-batch
            X_batch = X_train_shuffled[i:i+batch_size]
            y_batch = y_train_shuffled[i:i+batch_size]
            
            # Make predictions
            y_pred = predict(X_batch)
            
            # Compute the loss with L2 regularization (Ridge regression)
            loss = - (1 / batch_size) * np.sum(y_batch * np.log(y_pred) + (1 - y_batch) * np.log(1 - y_pred)) + (lambda_reg / 2) * np.sum(weights**2)
            
            # Compute gradients
            dw = (1 / batch_size) * np.dot(X_batch.T, (y_pred - y_batch)) + lambda_reg * weights  # L2 regularization term
            db = (1 / batch_size) * np.sum(y_pred - y_batch)
            
            # Update weights and bias
            weights -= learning_rate * dw
            bias -= learning_rate * db
        
        # Optionally, print the loss every 100 epochs
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss}")


In [24]:
# Hyperparameters
batch_size = 32  
lambda_reg = 0.1  # Regularization strength


In [25]:
# Train with mini-batch gradient descent and L2 regularization
train_logistic_regression_with_mini_batch(X_train, y_train, learning_rate, epochs, batch_size, lambda_reg)

Epoch 0, Loss: 0.3529024735107809
Epoch 100, Loss: 0.3003317211363782
Epoch 200, Loss: 0.31852724289431744
Epoch 300, Loss: 0.34408031095714003
Epoch 400, Loss: 0.31594507302455005
Epoch 500, Loss: 0.31977651652385514
Epoch 600, Loss: 0.34316128496102555
Epoch 700, Loss: 0.33046523584612186
Epoch 800, Loss: 0.40237135076543157
Epoch 900, Loss: 0.23110295257355093


In [26]:
# Evaluate the model
y_pred_test = predict(X_test)
y_pred_test = (y_pred_test > 0.5).astype(int)  # Convert to binary classification (0 or 1)
accuracy = accuracy_score(y_test, y_pred_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 65.44%


In [60]:
# CNN Model for Image Classification

# Reshape data to match CNN input (N, 64, 64, 1)
X_train_cnn = X_train.reshape(-1, 64, 64, 1)
X_test_cnn = X_test.reshape(-1, 64, 64, 1)


In [61]:
# One-hot encode labels
y_train_cnn = to_categorical(y_train, num_classes=2)
y_test_cnn = to_categorical(y_test, num_classes=2)


In [62]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Build the CNN model
model = Sequential([
    Conv2D(64, kernel_size=(3, 3), activation='relu', input_shape=(64, 64, 1)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),  # Dropout to prevent overfitting
    Dense(2, activation='softmax')  # Output layer for binary classification
])

C:\Users\subas\anaconda3\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [63]:
from tensorflow.keras.optimizers import Adam

# Compile the CNN model
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])



In [64]:
# Train the CNN model
model.fit(X_train_cnn, y_train_cnn, epochs=10, batch_size=32, validation_data=(X_test_cnn, y_test_cnn))


Epoch 1/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 20s 203ms/step - accuracy: 0.6225 - loss: 0.7252 - val_accuracy: 0.6558 - val_loss: 0.6524
Epoch 2/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 16s 188ms/step - accuracy: 0.6679 - loss: 0.6382 - val_accuracy: 0.6558 - val_loss: 0.6398
Epoch 3/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 16s 187ms/step - accuracy: 0.6494 - loss: 0.6393 - val_accuracy: 0.6647 - val_loss: 0.6347
Epoch 4/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 17s 195ms/step - accuracy: 0.6728 - loss: 0.6226 - val_accuracy: 0.6662 - val_loss: 0.6259
Epoch 5/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 19s 225ms/step - accuracy: 0.6969 - loss: 0.5970 - val_accuracy: 0.6617 - val_loss: 0.6335
Epoch 6/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 19s 221ms/step - accuracy: 0.6959 - loss: 0.5763 - val_accuracy: 0.6677 - val_loss: 0.6335
Epoch 7/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 16s 189ms/step - accuracy: 0.7303 - loss: 0.5387 - val_accuracy: 0.6632 - val_loss: 0.6436
Epoch 8/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 16s 187ms/step - accuracy: 0.7285 - loss: 0.5352 - val_accu

In [ ]:
# Evaluate the CNN model
loss, accuracy = model.evaluate(X_test_cnn, y_test_cnn)
print(f'Test Accuracy of CNN: {accuracy * 100:.2f}%')


In [40]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

datagen.fit(X_train_cnn)

# Use the data generator in model training
model.fit(datagen.flow(X_train_cnn, y_train_cnn, batch_size=32), epochs=15, validation_data=(X_test_cnn, y_test_cnn))


Epoch 1/15


C:\Users\subas\anaconda3\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


85/85 ━━━━━━━━━━━━━━━━━━━━ 17s 184ms/step - accuracy: 0.6453 - loss: 0.6986 - val_accuracy: 0.6677 - val_loss: 0.6286
Epoch 2/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 15s 177ms/step - accuracy: 0.6673 - loss: 0.6340 - val_accuracy: 0.6721 - val_loss: 0.6286
Epoch 3/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 17s 191ms/step - accuracy: 0.6495 - loss: 0.6442 - val_accuracy: 0.6721 - val_loss: 0.6314
Epoch 4/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 15s 178ms/step - accuracy: 0.6631 - loss: 0.6297 - val_accuracy: 0.6736 - val_loss: 0.6242
Epoch 5/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 16s 187ms/step - accuracy: 0.6535 - loss: 0.6400 - val_accuracy: 0.6736 - val_loss: 0.6216
Epoch 6/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 17s 200ms/step - accuracy: 0.6641 - loss: 0.6231 - val_accuracy: 0.6647 - val_loss: 0.6259
Epoch 7/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 17s 196ms/step - accuracy: 0.6516 - loss: 0.6324 - val_accuracy: 0.6499 - val_loss: 0.6272
Epoch 8/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 16s 188ms/step - accuracy: 0.6405 - loss: 0.6417 - val_accuracy: 0.669

In [54]:
from sklearn.metrics import classification_report

# Predict on the test set
y_pred = model.predict(X_test_cnn)
y_pred_classes = np.argmax(y_pred, axis=1)

# Classification report
print(classification_report(np.argmax(y_test_cnn, axis=1), y_pred_classes))


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step
              precision    recall  f1-score   support

           0       0.68      0.93      0.79       444
           1       0.58      0.18      0.28       233

    accuracy                           0.67       677
   macro avg       0.63      0.56      0.53       677
weighted avg       0.65      0.67      0.61       677



In [55]:
from tensorflow.keras.models import load_model

# Function to load and preprocess the new image
def load_and_preprocess_image(image_path):
    # Load the image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)  # Load in grayscale
    img = cv2.resize(img, (64, 64))  # Resize to match the input shape of the model
    img = img / 255.0  # Normalize the pixel values to [0, 1]
    
    # Reshape the image to match the input format of the CNN (64, 64, 1)
    img = img.reshape(1, 64, 64, 1)
    
    return img



In [56]:
# Predict the new image
def predict_new_image(image_path):
    # Preprocess the new image
    img = load_and_preprocess_image(image_path)
    
    # Make a prediction
    prediction = model.predict(img)
    
    # Get the class with the highest probability (since it's a binary classification)
    predicted_class = np.argmax(prediction, axis=1)[0]
    
    # Map the predicted class to a descriptive statement
    class_labels = {
        0: "The image suggests no signs of breast cancer.",  # Adjust based on your dataset
        1: "The image indicates potential signs of breast cancer. Please consult a doctor for further analysis."  # Adjust based on your dataset
    }
    predicted_label = class_labels[predicted_class]
    
    print(f"Predicted class: {predicted_label}")
    return predicted_label

In [59]:
image_path = r"D:\Medical imaging diagnostics(breast cancer)\33_995020214_png.rf.4612f989168551a8c7c4145bf28bde1f.jpg"  # Update with the actual image path
predicted_label = predict_new_image(image_path)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Predicted class: The image suggests no signs of breast cancer.
